## Lab 8: Check the claims ## 

Today we asked can we trust the data? We looked at what is missing, who is not in the table, and
where the data comes from. In this lab you take the role of the person who reads someone else's
claim and has to decide if it is true. You get four sentences that someone could write about the
Airbnb table. For each sentence you test it with data, you decide a verdict, and you write a new
sentence that you could defend.

The four possible verdicts are: 
- Holds, 
- Holds with reservations, 
- Does not hold, and 
- Cannot be concluded from this table. 

First you see one claim that is solved as an example, with the code, the reasoning, 
the verdict and the new sentence. Then you solve the four claims in the same way. 

Write the verdict and your new sentence in a markdown cell below each test, in your own words. 
A good new sentence says how many are behind a number and what it is about.
For every claim, ask yourself one question: what does the sentence not say?

Use the file listings_clean.csv (in your data folder), and work in a new notebook. 
Everything you need is something we have used before. 
Tasks 1 to 4 are claims that you test. 
Task 5 is different: it is a short text about the whole table, with no verdict. 
If you get stuck: read the error message from the bottom up, print the thing you are working with, 
and look at the lecture notebooks. Then ask.

### Four claims, and a data note ###

Tools you may need

listings = pd.read_csv('data/listings_clean.csv',
parse_dates=['first_review', 'last_review'])
column.mean() column.median() column.nunique()
column.notna().sum() column.isna().mean() * 100
column.value_counts().sort_index()
(a & b).sum() # & means AND, for True/False columns
column.dt.year # the year of a date column
table.groupby('x')['y'].agg(['count', 'median'])
table.groupby('x').size() # how many rows there are in each group

### Example: how to check a claim ###

This is claim 0, solved from start to end. You solve the four claims below in the same way.
Claim 0: "The typical Stockholm listing has 36 reviews."

Step 1: check the number, and what is behind it. We calculate the mean and the median of
number_of_reviews, and we count the listings without any review and the listings with more than
100.

reviews = listings['number_of_reviews']

print(reviews.mean().round(1), reviews.median())
print((reviews == 0).sum(), ((reviews == 0).mean() * 100).round(1))
print((reviews > 100).sum())

# 35.8 9.0
# 644 14.1
# 426

Step 2: read the result. The number 36 is right, but only as a mean. The word "typical" points to
the median, and that is 9. The mean is higher because 426 listings have more than 100 reviews and
pull it up. And 644 listings, 14 percent, have no review at all, which the sentence does not mention.
So 36 is not what a typical listing has.

Verdict: Does not hold. The number is correct, but the word "typical" is wrong.
New sentence: "The median Stockholm listing has 9 reviews, and 14 percent have none. The mean
is 36, because a few hundred listings have more than 100 reviews."

In [3]:
import pandas as pd

listings = pd.read_csv(
    'data/listings_clean.csv',
    parse_dates=['first_review', 'last_review']
)

print("Shape:", listings.shape)

Shape: (4574, 73)


# Tasks

### 1 Claim 1: "A night in Stockholm costs 1,791."

In [4]:
price = listings['price']

# 1. Number of listings with and without a price + median
print("With price:", price.notna().sum())
print("Without price:", price.isna().sum(), f"({price.isna().mean() * 100:.1f}%)")
print("Median price:", price.median().round(1))

# 2. What do priceless ads have in common (availability_365)
no_price = listings[price.isna()]
print("No-price listings with 0 open days:", (no_price['availability_365'] == 0).sum(),
      f"({(no_price['availability_365'] == 0).mean() * 100:.1f}%)")

# 3. Long term (>= 28 nights) vs short term (< 28 nights) among price listings
with_price = listings[price.notna()]
long_stays = with_price[with_price['minimum_nights'] >= 28]
short_stays = with_price[with_price['minimum_nights'] < 28]

print("Long stays (>= 28 nights) count:", len(long_stays), "| median:", long_stays['price'].median().round(1))
print("Short stays (< 28 nights) count:", len(short_stays), "| median:", short_stays['price'].median().round(1))

With price: 2957
Without price: 1617 (35.4%)
Median price: 1791.0
No-price listings with 0 open days: 1490 (92.1%)
Long stays (>= 28 nights) count: 97 | median: 708.9
Short stays (< 28 nights) count: 2856 | median: 1844.2


### Task 1 - Reasoning:

The number 1,791 (in SEK) is indeed the median price, so the figure itself is accurate. 
However, it is the median of only the 2,957 listings that have a listed price (64.6%), 
while 1,617 listings (35.4%) have no price at all—and most of those have zero open days (availability_365 == 0). 
Furthermore, mixing different stay lengths lowers the overall median: 
97 priced listings require 28 nights or more with a median of 708.9 SEK per night, 
whereas the remaining 2,860 short-stay listings have a higher median of 1,839.2 SEK.

### Verdict: Holds with reservations.

### New sentence: 
"On 26 and 27 September 2026, the median price per night was 1,791 SEK 
for the 2,957 Airbnb listings in Stockholm that had a price, 
which is 65 percent of all listings 
(while 35 percent had no price listed and mostly zero open days)."

### 2 Claim 2: "The listings with a perfect 5.00 rating are the best in Stockholm."

In [5]:
# Classified ads and those with a perfect 5.00
rated = listings[listings['review_scores_rating'].notna()]
perfect = rated[rated['review_scores_rating'] == 5.0]
others = rated[rated['review_scores_rating'] < 5.0]

print("Rated listings:", len(rated))
print("Perfect 5.00 listings:", len(perfect), f"({len(perfect) / len(rated) * 100:.1f}%)")

# How many of the perfect ones have at least 20 reviews, and how many less than 20?
perfect_20plus = (perfect['number_of_reviews'] >= 20).sum()
print("Perfect with >= 20 reviews:", perfect_20plus, f"({perfect_20plus / len(perfect) * 100:.1f}%)")
print("Perfect with < 20 reviews:", (perfect['number_of_reviews'] < 20).sum(),
      f"({(perfect['number_of_reviews'] < 20).mean() * 100:.1f}%)")

# Median number of reviews for 5.00 vs others
print("Median reviews (5.00):", perfect['number_of_reviews'].median())
print("Median reviews (< 5.00):", others['number_of_reviews'].median())

Rated listings: 3930
Perfect 5.00 listings: 1484 (37.8%)
Perfect with >= 20 reviews: 130 (8.8%)
Perfect with < 20 reviews: 1354 (91.2%)
Median reviews (5.00): 4.0
Median reviews (< 5.00): 25.0


### Task 2 - Reasoning:

Out of 3,930 listings that have a rating, 1,484 (37.8%) have a perfect 5.00. 
However, only 130 of those 5.00 listings (8.8%) have at least 20 reviews—meaning 91.2% 
of the perfect listings have fewer than 20 reviews. 
The median number of reviews is just 4 for the 5.00 listings, 
compared to 25 for listings rated below 5.00, so a perfect score is mostly a reflection 
of a tiny sample of reviews rather than proof of being the best listing.

### Verdict: Does not hold.

### New sentence: 
"Out of 3,930 rated listings in Stockholm, 38 percent (1,484 listings) have a perfect 5.00 rating, 
but 91 percent of them have fewer than 20 reviews (with a median of 4 reviews), 
so the rating says little about which listings are actually best."

### 3 Claim 3: "The market has grown: only 131 listings started in 2016, but 610 in 2024."

In [6]:
# Number of ads per year of first review (first_review)
first_year = listings['first_review'].dt.year
last_year = listings['last_review'].dt.year

print("2016 first_review count:", (first_year == 2016).sum())
print("2024 first_review count:", (first_year == 2024).sum())

# Share of ads from 2016 and 2024 that had a review in 2026.
in_2016 = listings[first_year == 2016]
in_2024 = listings[first_year == 2024]

print("Share reviewed in 2026 (2016 cohort):", (in_2016['last_review'].dt.year == 2026).mean().round(2))
print("Share reviewed in 2026 (2024 cohort):", (in_2024['last_review'].dt.year == 2026).mean().round(2))

2016 first_review count: 131
2024 first_review count: 610
Share reviewed in 2026 (2016 cohort): 0.53
Share reviewed in 2026 (2024 cohort): 0.67


### Task 3 - Reasoning:

The counts in the table match the claim (131 listings had their first_review in 2016 and 610 in 2024), 
and among those surviving listings, 53% (0.53) from 2016 and 67% (0.67) from 2024 were reviewed again in 2026. 

However, this table is a single snapshot of listings that still exist today—any listing that started in 2016 and later closed was removed before the scrape date and is missing from the dataset (survivorship bias), plus first_review does not capture unreviewed listings. Therefore, these numbers cannot show how much the market actually grew.

### Verdict: Cannot be concluded from this table.

### New sentence: 
"Of the Stockholm Airbnb listings that still existed on 26 and 27 September 2026, 
131 received their first review in 2016 and 610 in 2024, 
but we cannot say how many listings actually started in those years because closed listings are missing."

### 4 Claim 4: "Most hosts in Stockholm are professional companies with many listings."

In [7]:
# Unique hosts (host_id) and number of ads per host
host_sizes = listings.groupby('host_id').size()

print("Total listings:", len(listings))
print("Unique hosts:", listings['host_id'].nunique())
print("Hosts with 1 listing:", (host_sizes == 1).sum(), f"({(host_sizes == 1).mean() * 100:.1f}%)")
print("Hosts with 2+ listings:", (host_sizes >= 2).sum(), f"({(host_sizes >= 2).mean() * 100:.1f}%)")

# How many listings do hosts with 2+ listings hold together and what is the maximum?
multi_listings = host_sizes[host_sizes >= 2].sum()
print("Listings owned by 2+ hosts:", multi_listings, f"({multi_listings / len(listings) * 100:.1f}%)")
print("Largest host listings count:", host_sizes.max())

# Checking whether there is a column on whether the host is a company
print("Host columns:", [c for c in listings.columns if 'host' in c])

Total listings: 4574
Unique hosts: 3591
Hosts with 1 listing: 3237 (90.1%)
Hosts with 2+ listings: 354 (9.9%)
Listings owned by 2+ hosts: 1337 (29.2%)
Largest host listings count: 49
Host columns: ['host_id', 'host_profile_id', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_is_superhost', 'host_listings_count', 'host_has_profile_pic', 'host_identity_verified', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms']


### Task 4 - Reasoning:

Across all 4,574 listings there are 3,591 unique hosts, and 3,237 of them (90.1%) have only a single listing. 
Only 354 hosts (9.9%) manage two or more listings—owning 1,337 listings (29.2% of the table) combined, 
up to a maximum of 49 listings for the largest host—and there is no column in the dataset 
indicating whether any host is a registered company rather than a private individual.

### Verdict: Does not hold (and the "company" part cannot be concluded from this table).

### New sentence: 
"Out of 3,591 unique Airbnb hosts across 4,574 Stockholm listings, 90 percent (3,237 hosts) have only one listing, 
while the 10 percent (354 hosts) with two or more listings own 29 percent (1,337) of all listings."

5 Task 5 is different: a data note, not a claim

There is nothing to test here, and no verdict. Instead you write about the whole table,
using what you found in claims 1 to 4. Write a short text in a markdown cell, at most ten
sentences, as if it was the first paragraph of a report about this table. Answer in your own
words: what is missing (with a number)? Who or what is not in the table? Who made the
data, when, and under which licence? Which columns are calculated and not measured?
And finally: what can this table not tell us? Finish with one sentence: which of the four
claims would you most want to correct if you read it in a newspaper, and why?

In [8]:
print("Missing %:")
print((listings.isna().mean() * 100).round(1)[lambda s: s > 0])
print("\nestimated_ columns:", [c for c in listings.columns if c.startswith('estimated_')])

Missing %:
source                         62.7
description                     1.2
bathrooms_text                  0.2
beds                            2.2
price                          35.4
price_quote_checkin_date       34.0
price_quote_checkout_date      34.0
price_quote_total_price        35.4
price_quote_price_per_night    35.4
price_quote_raw                34.0
minimum_nights                  0.1
maximum_nights                  0.1
minimum_minimum_nights          0.1
maximum_minimum_nights          0.1
minimum_maximum_nights          0.1
maximum_maximum_nights          0.1
has_availability                1.0
estimated_revenue_l365d        35.4
first_review                   14.1
last_review                    14.1
review_scores_rating           14.1
review_scores_accuracy         14.1
review_scores_cleanliness      14.1
review_scores_checkin          14.1
review_scores_communication    14.1
review_scores_location         14.1
review_scores_value            14.1
reviews_per_month

### Task 5 - Resoning:

- This report examines a dataset of 4,574 Airbnb listings in Stockholm collected as a snapshot on 26 and 27 September 2026. 

- Significant portions of the table are missing: 35 percent of listings (`1,617`) have no listed price—mostly inactive listings with zero open days—and 14 percent (`644`) lack a rating because they have never been reviewed. 

- Because the data comes from a single public web scrape, it excludes listings that closed before September 2026, as well as hotels and rentals offered on other platforms. 

- In addition, columns starting with `estimated_` (such as `estimated_occupancy_l365d` and `estimated_revenue_l365d`), along with fields like `days_since_review` and `calculated_host_listings_count`, are calculated model estimates rather than directly measured bookings. 

- Consequently, the table cannot tell us why a host set a particular price, how many nights were truly booked, or whether guests were genuinely satisfied beyond star ratings. 

- If I read one of the four claims in a newspaper, I would most want to correct **Claim 3** (*"The market has grown: only 131 listings started in 2016, but 610 in 2024"*), because comparing surviving listings from ten years ago against recent ones ignores all closed listings in between and tricks readers into mistaking survivorship bias for market growth.